
# 02 数据工程与 PIT

> **本章要解决的问题**：供应商发来一份“已复权、已对齐、已更新”的数据，为什么回测还是赚了未来？同一份财报为什么在三月和七月看到两个值？期货换月、夜盘成交、停牌涨跌停，又是怎么把一张干净的价格表变成一堆陷阱的？

## 本章导读

第 28 章讲过“库存修订与当时可见的数据”，第 40 章讲过因子数据库的 bitemporal 表结构。这一章把视角放到**数据进入流水线之前**：供应商给的原始数据长什么样、哪里可能出错、怎么在入库前把它问清楚。

数据工程要守住的四件事：

1. **口径**：字段含义、单位、复权方式、时区与交易日归属，写进数据字典；
2. **PIT**：每条数据带 `knowledge_date`，回测只看当时可见的版本；
3. **连续性**：复权让价格连续、换月让期货连续、夜盘归属让交易日连续；
4. **可信度**：质量断言、存活偏差检查、版本快照，让“这份数据能不能用”有据可查。

一句话：**数据工程不是“把数据读进来”，而是“把数据的每一处不确定性写清楚”**。

### 学习目标

- 给定一份供应商数据说明，能写出最小数据字典（字段、单位、主键、时间语义、复权口径、更新频率）。
- 给定带修订的财务数据，能用 `knowledge_date` 与 `merge_asof` 做 as-of 查询，并识别“用最新版本”造成的前视。
- 给定价格与除权事件，能算后复权/前复权价格与复权成交量，验证复权后收益连续。
- 给定期货多合约行情，能按持仓量选主力、按规则换月、拼出连续收益，并说清直接拼接为什么会假跳空。
- 给定带时间戳的夜盘成交，能把它归属到正确的交易日。
- 给定一张含停牌/涨跌停/退市的面板，能算存活偏差、标记不可交易样本，并跑质量断言与版本快照。

### 先修

第 28 章的 PIT 与库存修订；第 40 章的双时间轴表结构；pandas 的 `merge_asof`、`groupby`；Python 标准库 `sqlite3`、`hashlib`、`json`。

## 在量化研究中的位置

- **谁用**：数据工程、因子研究员、回测平台、风控与合规。
- **输入**：供应商行情/财务/期货数据、数据字典、除权事件、合约规则、交易日历。
- **输出**：PIT 面板、复权价格、连续合约、可交易标记、质量报告、版本快照。
- **在流程的哪一步**：数据入库之前、因子计算之前；所有下游研究共同的地基。
- **失败模式**：用最新版财务数据回测、复权口径混用、换月直接拼接、夜盘算错交易日、存活偏差、停牌被当正常交易。
- **最小工作流**：读字典 → 建 PIT 表 → 复权/换月/夜盘 → 可交易标记 → 质量断言 → 版本快照。

### 环境准备



In [1]:

import hashlib
import io
import json
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
print("pandas", pd.__version__, "| sqlite", sqlite3.sqlite_version)


pandas 3.0.2 | sqlite 3.51.2



## 1. 数据字典：先约定字段，再写代码

数据字典不是“字段列表”，而是一份**约定**。同一列 `close`，可能是收盘价、结算价、不复权价、后复权价；同一列 `date`，可能是自然日、交易日、成交日或结算日。口径没写清，后面每一行代码都在猜。

最小数据字典至少包含：

| 项目 | 例子 | 不写会怎样 |
|---|---|---|
| 字段名与单位 | `close`：元/吨，后复权 | 把每吨报价当一手价格 |
| 主键 | `(asset, trade_date)` | 同一行重复入库 |
| 时间语义 | `trade_date` 交易日，`knowledge_date` 可见日 | 偷看未来 |
| 复权口径 | 后复权 / 不复权 + 复权因子 | 收益断裂 |
| 更新频率 | 日频，T+1；财务 T+1 季度 | 快照被覆盖 |
| 缺失与异常 | 停牌不写行；涨跌停照写 | 把停牌当 0 收益 |
| 交易日历 | 夜盘归属、节假日 | 夜盘算错日期 |

本章后面每一节，其实都在补数据字典里的某一格。

---

## 2. PIT 与修订：当时能看到什么

财务数据是典型的“同一事实、多个版本”：2023 年报在 2024 年 3 月首次披露（ROE 12%），6 月因追溯调整改为 11%。如果回测在 2024 年 4 月用 11%，就用了当时还不存在的数。

PIT（point-in-time）的做法是给每条记录加一列 `knowledge_date`：这条数据从哪一天开始可见。查询时只取 `knowledge_date <= 回测日` 的最新一条。pandas 里用 `merge_asof(direction="backward")` 就能做这个 as-of join。

### 2.1 代码：CSV 装载 + merge_asof



In [2]:

csv_text = """asset,report_period,roe,knowledge_date,snapshot_id
A001,2023-12-31,0.120,2024-03-20,snap-01
A001,2023-12-31,0.110,2024-06-15,snap-02
A002,2023-12-31,0.090,2024-03-25,snap-01
A002,2023-12-31,0.096,2024-08-10,snap-03
A003,2023-12-31,0.075,2024-04-02,snap-01
"""
reports = pd.read_csv(io.StringIO(csv_text))
reports["knowledge_date"] = pd.to_datetime(reports["knowledge_date"])
print("原始记录", len(reports), "条；每资产修订次数：")
print(reports.groupby("asset")["knowledge_date"].agg(["count", "min", "max"]).to_string())

query = pd.DataFrame({
    "asset": ["A001", "A002", "A003", "A001", "A002", "A003"],
    "as_of": ["2024-04-01", "2024-04-01", "2024-04-01", "2024-07-01", "2024-07-01", "2024-07-01"],
})
query["as_of"] = pd.to_datetime(query["as_of"])
panel = pd.merge_asof(
    query.sort_values("as_of"), reports.sort_values("knowledge_date"),
    left_on="as_of", right_on="knowledge_date", by="asset", direction="backward",
)
print(panel[["asset", "as_of", "roe", "knowledge_date", "snapshot_id"]].to_string(index=False))

latest = reports.sort_values("knowledge_date").groupby("asset").tail(1)
wrong = query.merge(latest[["asset", "roe"]], on="asset", how="left")
print("若用最新版本（错误做法）在 2024-04-01 看到的 roe：")
print(wrong[wrong["as_of"] == "2024-04-01"][["asset", "as_of", "roe"]].to_string(index=False))


原始记录 5 条；每资产修订次数：
       count        min        max
asset                             
A001       2 2024-03-20 2024-06-15
A002       2 2024-03-25 2024-08-10
A003       1 2024-04-02 2024-04-02
asset      as_of   roe knowledge_date snapshot_id
 A001 2024-04-01 0.120     2024-03-20     snap-01
 A002 2024-04-01 0.090     2024-03-25     snap-01
 A003 2024-04-01   NaN            NaT         NaN
 A001 2024-07-01 0.110     2024-06-15     snap-02
 A002 2024-07-01 0.090     2024-03-25     snap-01
 A003 2024-07-01 0.075     2024-04-02     snap-01
若用最新版本（错误做法）在 2024-04-01 看到的 roe：
asset      as_of   roe
 A001 2024-04-01 0.110
 A002 2024-04-01 0.096
 A003 2024-04-01 0.075



**读数**：2024-04-01 时 A001 只能看到 3 月 20 日披露的 0.120，A003 因为 4 月 2 日才披露还是缺失；到 2024-07-01，A001 被修订为 0.110。如果图省事直接取每个资产的最新值，在 4 月 1 日就会看到 A001 的 0.110、A002 的 0.096、A003 的 0.075——**三个数里两个还没发生，一个甚至还没披露**，这就是典型的 PIT 前视。

**常见错误**：

1. 只保留最新版本，历史版本被覆盖。
2. 用 `report_period` 排序而不是 `knowledge_date`——报告期早不等于可见时间早。
3. 对修订数据不做标记，回测和实盘看到的口径不一致。

---

## 3. 落在数据库里：sqlite 的 as-of 查询

数据量大或多人共用时，PIT 面板通常落在数据库里。和第 40 章一样，表的主键包含 `knowledge_date`，as-of 查询分两步：先按 `knowledge_date <= as_of` 取每个业务键的最大知识日，再回表取那一行。

### 3.1 代码：建表、装载与 as-of



In [3]:

conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("""
CREATE TABLE fin (
    asset TEXT NOT NULL,
    report_period TEXT NOT NULL,
    roe REAL NOT NULL,
    knowledge_date TEXT NOT NULL,
    snapshot_id TEXT NOT NULL,
    PRIMARY KEY (asset, report_period, knowledge_date)
)
""")
cur.executemany(
    "INSERT INTO fin VALUES (?,?,?,?,?)",
    [(r.asset, r.report_period, r.roe, r.knowledge_date.strftime("%Y-%m-%d"), r.snapshot_id)
     for r in reports.itertuples()],
)
conn.commit()

ASOF_SQL = """
WITH latest AS (
    SELECT asset, report_period, MAX(knowledge_date) AS kd
    FROM fin
    WHERE knowledge_date <= ? AND report_period <= ?
    GROUP BY asset, report_period
)
SELECT f.asset, f.report_period, f.roe, f.knowledge_date
FROM fin AS f
JOIN latest AS l
  ON f.asset = l.asset AND f.report_period = l.report_period AND f.knowledge_date = l.kd
ORDER BY f.asset
"""
for as_of in ("2024-04-01", "2024-07-01"):
    rows = cur.execute(ASOF_SQL, (as_of, as_of)).fetchall()
    print(f"as-of {as_of}：", rows)
print("记录数", cur.execute("SELECT COUNT(*) FROM fin").fetchone()[0],
      "| 资产数", cur.execute("SELECT COUNT(DISTINCT asset) FROM fin").fetchone()[0])


as-of 2024-04-01： [('A001', '2023-12-31', 0.12, '2024-03-20'), ('A002', '2023-12-31', 0.09, '2024-03-25')]
as-of 2024-07-01： [('A001', '2023-12-31', 0.11, '2024-06-15'), ('A002', '2023-12-31', 0.09, '2024-03-25'), ('A003', '2023-12-31', 0.075, '2024-04-02')]
记录数 5 | 资产数 3



**读数**：as-of 2024-04-01 只返回两条记录（A001 的 0.120、A002 的 0.090），A003 因为披露日晚于查询日而不可见；as-of 2024-07-01 返回三条，A001 已经是修订后的 0.110。注意查询里同时过滤了 `knowledge_date` 和 `report_period`——**知识时间和有效时间两条轴都要管**。

**常见错误**：

1. 只过滤 `knowledge_date`，把回测日之后才发生的“报告期”也放进来。
2. 用 `MAX(roe)` 之类的聚合代替“取最新一行”，把修订值和取值混为一谈。
3. 数据库里不留历史行，修订只能 UPDATE 覆盖。

---



## 4. 复权：价格连续，收益才连续

除权除息那天，股价会“凭空”跳一下：10 送 10，价格从 10 元变 5 元。这不是亏损，是股本变了。不做复权，算出来的收益是 −50%；做了复权，收益才是真实的 +0.39%。

### 4.1 前复权与后复权

- **后复权**：以最早价格为基准，把除权后的价格向上乘因子。优点是**历史价格永不改变**，适合回测与长期收益计算；
- **前复权**：以最新价格为基准，把历史价格向下调整。优点是和当前盘面价格对齐，适合画图与看盘；
- 两者只差一个整体缩放，**收益率完全一致**。

做研究要特别注意：前复权每次有新的分红送股都会重算历史，等于“用未来的除权信息调整过去的价格”，回测里用它容易引入 PIT 问题；后复权数据一旦生成就不再变化，更适合做版本化存储。

### 4.2 复权因子与成交量

复权因子由供应商给出，也可以从除权事件自己算。价格乘因子，**成交量要反向除因子**：除权后股数翻倍，成交股数也翻倍，要折回除权前的口径才可比。

### 4.3 代码：一次 2:1 拆股



In [4]:

raw = np.array([10.00, 10.20, 5.12, 5.20, 5.30])
af = np.array([1.0, 1.0, 2.0, 2.0, 2.0])
post = raw * af
pre = post * (raw[-1] / post[-1])
adj = pd.DataFrame({
    "原始价": raw, "后复权价": np.round(post, 4), "前复权价": np.round(pre, 4),
    "原始收益": pd.Series(raw).pct_change(), "后复权收益": pd.Series(post).pct_change(),
})
print(adj.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print("前复权与后复权收益最大差异：",
      round(float(np.nanmax(np.abs(np.diff(pre) / pre[:-1] - np.diff(post) / post[:-1]))), 10))
raw_vol = np.array([1000, 1200, 2400, 2600, 2800])
print("成交量（股）：原始", raw_vol.tolist(),
      "| 后复权（折成除权前股数）", (raw_vol / af).astype(int).tolist())


    原始价    后复权价   前复权价    原始收益  后复权收益
10.0000 10.0000 5.0000     NaN    NaN
10.2000 10.2000 5.1000  0.0200 0.0200
 5.1200 10.2400 5.1200 -0.4980 0.0039
 5.2000 10.4000 5.2000  0.0156 0.0156
 5.3000 10.6000 5.3000  0.0192 0.0192
前复权与后复权收益最大差异： 0.0
成交量（股）：原始 [1000, 1200, 2400, 2600, 2800] | 后复权（折成除权前股数） [1000, 1200, 1200, 1300, 1400]



**读数**：第 3 天 2:1 拆股，原始价从 10.20 掉到 5.12，原始收益 −49.80%，看起来像暴跌；后复权价从 10.20 到 10.24，收益 +0.39%；前复权价从 5.10 到 5.12，收益同样是 +0.39%。前复权与后复权的收益最大差异是 0——**复权方式只影响价格刻度，不影响收益**。成交量按因子反向调整后，2,400 股折回除权前口径是 1,200 股。

**常见错误**：

1. 用前复权数据做长时间回测——除权信息随未来变化，隐含前视。
2. 只复权价格、不复权成交量，换手率与成交额全错。
3. 复权因子用加法而不是乘法，多次除权后偏差越来越大。

---

## 5. 期货换月与夜盘归属

### 5.1 换月：连续合约不是“拼价格”

期货合约有到期日，主力合约会从近月切到远月。直接拿“每天的主力合约收盘价”拼成一条曲线，换月那天会出现一个**假跳空**：旧合约 3800、新合约 3755，看起来一天跌了 1.18%，其实两个合约价差本来就有 1.44%。

正确做法有两种：

1. **收益率拼接**：每天只取“当天主力合约”的日收益，再累乘成净值；
2. **价格比率调整**：换月时把新合约的历史价格乘以（旧合约价 / 新合约价），让拼接点连续。

换月还带来**展期收益/成本**：远近月价差、换月频率、换月规则（持仓量、成交量、固定日期）都会影响连续合约的表现。

### 5.2 夜盘归属：成交时间 ≠ 交易日

国内商品期货有夜盘，21:00 之后成交的，算**下一个交易日**；凌晨 2:30 之前的，属于前一晚夜盘。周五晚上的夜盘归下周一。如果按自然日归属，隔夜收益、开盘缺口、持仓天数全都会错。

### 5.3 代码：主力切换与夜盘归属



In [5]:

dates = pd.bdate_range("2024-01-02", periods=6)
prices = pd.DataFrame({
    "rb2403": [3800, 3810, 3820, 3830, 3835, 3840],
    "rb2405": [3750, 3755, 3760, 3768, 3775, 3782],
    "rb2410": [3700, 3702, 3705, 3708, 3712, 3715],
}, index=dates)
oi = pd.DataFrame({
    "rb2403": [120000, 95000, 60000, 20000, 8000, 3000],
    "rb2405": [80000, 110000, 150000, 180000, 200000, 210000],
    "rb2410": [30000, 32000, 35000, 38000, 40000, 42000],
}, index=dates)
main = oi.idxmax(axis=1)
levels, rets = [1.0], [np.nan]
for t in range(1, len(dates)):
    c = main.iloc[t]
    r = prices.loc[dates[t], c] / prices.loc[dates[t - 1], c] - 1
    rets.append(r)
    levels.append(levels[-1] * (1 + r))
roll = pd.DataFrame({
    "交易日": dates.strftime("%Y-%m-%d"), "主力合约": main.values,
    "收盘价": [prices.loc[d, c] for d, c in zip(dates, main)],
    "持仓量": [oi.loc[d, c] for d, c in zip(dates, main)],
    "日收益": [round(x, 5) if not np.isnan(x) else np.nan for x in rets],
    "连续净值": [round(x, 6) for x in levels],
    "换月": ((main != main.shift(1)) & main.shift(1).notna()).values,
})
print(roll.to_string(index=False))
roll_day = roll.index[roll["换月"]][0]
old_c, new_c = main.iloc[roll_day - 1], main.iloc[roll_day]
gap = prices.loc[dates[roll_day], new_c] / prices.loc[dates[roll_day], old_c] - 1
print(f"换月日 {dates[roll_day].date()}：{old_c} -> {new_c}，新旧合约价差 {gap:.2%}")
naive = [round(float(prices.loc[d, c]), 1) for d, c in zip(dates, main)]
print("连续净值若直接拼接原始价（会有假跳空）：", naive)


def to_trade_date(ts):
    ts = pd.Timestamp(ts)
    if ts.hour >= 21:
        return (ts.normalize() + pd.offsets.BDay(1)).strftime("%Y-%m-%d")
    if ts.hour < 3:
        return ((ts.normalize() - pd.Timedelta(days=1)) + pd.offsets.BDay(1)).strftime("%Y-%m-%d")
    return ts.normalize().strftime("%Y-%m-%d")


ts_list = ["2024-01-05 21:30", "2024-01-06 01:30", "2024-01-03 01:30", "2024-01-04 09:30"]
print("夜盘归属：21:00 后算下一交易日；凌晨 3:00 前算前一晚夜盘所属交易日")
print(pd.DataFrame({"成交时间": ts_list, "归属交易日": [to_trade_date(t) for t in ts_list]}).to_string(index=False))


       交易日   主力合约  收盘价    持仓量     日收益     连续净值    换月
2024-01-02 rb2403 3800 120000     NaN 1.000000 False
2024-01-03 rb2405 3755 110000 0.00133 1.001333  True
2024-01-04 rb2405 3760 150000 0.00133 1.002667 False
2024-01-05 rb2405 3768 180000 0.00213 1.004800 False
2024-01-08 rb2405 3775 200000 0.00186 1.006667 False
2024-01-09 rb2405 3782 210000 0.00185 1.008533 False
换月日 2024-01-03：rb2403 -> rb2405，新旧合约价差 -1.44%
连续净值若直接拼接原始价（会有假跳空）： [3800.0, 3755.0, 3760.0, 3768.0, 3775.0, 3782.0]
夜盘归属：21:00 后算下一交易日；凌晨 3:00 前算前一晚夜盘所属交易日
            成交时间      归属交易日
2024-01-05 21:30 2024-01-08
2024-01-06 01:30 2024-01-08
2024-01-03 01:30 2024-01-03
2024-01-04 09:30 2024-01-04



**读数**：1 月 3 日主力从 rb2403 切到 rb2405，两合约同日价差 −1.44%；按收益率拼接的连续净值 6 天从 1.000000 走到 1.008533（+0.85%），换月当天的收益只取新合约自己的 +0.13%。若直接把两个合约的收盘价拼起来，3800 → 3755 会凭空多出一根 −1.18% 的假阴线。夜盘方面：周五 21:30 与周六 01:30 的成交都归属下周一 2024-01-08，周三凌晨 01:30 归属当天 2024-01-03，白天 09:30 归属当天。

**常见错误**：

1. 直接用主力合约价格拼接连续合约，换月日出现假跳空。
2. 近月合约一路持有到交割，不换月，也不处理交割风险。
3. 夜盘成交按自然日归属，隔夜收益和持仓天数都算错。

---



## 6. 停牌、涨跌停、存活偏差与数据质量

### 6.1 三张“隐形红旗”

价格表看起来干净，不代表能用。三类问题最隐蔽：

1. **停牌**：停牌日没有成交，若直接把它当 0 收益，组合权重会漂移，波动被低估；
2. **涨跌停**：涨停时买不进、跌停时卖不出，回测若照常成交，会系统性高估换手与收益；
3. **存活偏差**：用“今天的成分股”回测历史，等于只研究活下来的公司——退市、暴雷、被并购的股票被剔除，历史平均收益被抬高。本章小样本里，全样本平均日收益是 −0.91%，只用当前存活股票算出来是 +0.908%，偏差 +1.818%。

处理方式不是“删掉这些行”，而是加标记：`suspended`、`limit_up`、`limit_down`、`delisted`，再加一个汇总的 `tradable` 列。下游回测遇到不可交易样本，跳过成交而不是跳过收益。

### 6.2 质量断言与版本快照

质量断言要写成可执行的检查：字段完整、主键唯一、价格为正、无价格缺失、资产/日期覆盖数、复权因子为正。断言失败就不允许发布快照。

快照的做法和第 38 章一致：把面板按业务键排序、序列化成规范文本，算 sha256 取前 12 位作为 `snapshot_id`；manifest 记录行数、断言结果、代码版本。这样任何一批数据都能回答“这是哪一版、通过了哪些检查、谁生成的”。

### 6.3 代码：标记、偏差、断言、快照



In [6]:

panel2 = pd.DataFrame({
    "asset": ["A", "A", "A", "B", "B", "B", "C", "C", "D", "D"],
    "date": ["2024-01-02", "2024-01-03", "2024-01-04"] * 2 + ["2024-01-02", "2024-01-03"] + ["2024-01-02", "2024-01-03"],
    "close": [10.0, 10.2, 10.1, 20.0, 19.8, 20.2, 5.0, 4.5, 8.0, 8.2],
    "suspended": [False] * 10,
    "limit_up": [False, True, False, False, False, False, False, False, False, False],
    "limit_down": [False, False, False, False, True, False, False, False, False, False],
    "delisted": [False] * 6 + [True, True] + [False, False],
})
panel2["ret"] = panel2.groupby("asset")["close"].pct_change()
full = panel2["ret"].mean()
surv = panel2[~panel2["asset"].isin(["C"])]["ret"].mean()
print(f"全样本平均日收益 {full:.4%}；只用当前存活股票 {surv:.4%}；存活偏差 {surv - full:+.4%}")

panel2["tradable"] = ~(panel2["suspended"] | panel2["limit_up"] | panel2["limit_down"])
print("不可交易样本数", int((~panel2["tradable"]).sum()), "/", len(panel2))
print(panel2[~panel2["tradable"]][["asset", "date", "limit_up", "limit_down", "suspended"]].to_string(index=False))


def check_panel(df):
    checks = {
        "字段完整": set(["asset", "date", "close"]).issubset(df.columns),
        "主键唯一": not df.duplicated(["asset", "date"]).any(),
        "价格为正": bool((df["close"] > 0).all()),
        "无价格缺失": bool(df["close"].notna().all()),
        "资产数": int(df["asset"].nunique()),
        "日期数": int(df["date"].nunique()),
    }
    return checks


checks = check_panel(panel2)
print("质量断言：", checks)

canonical = panel2.sort_values(["asset", "date"]).to_csv(index=False)
snapshot_id = hashlib.sha256(canonical.encode("utf-8")).hexdigest()[:12]
manifest = {"snapshot_id": snapshot_id, "rows": len(panel2), "checks": checks,
            "code_version": "data-eng-0.1"}
print("版本快照 id：", snapshot_id)
print("manifest：", json.dumps(manifest, ensure_ascii=False))


全样本平均日收益 -0.9100%；只用当前存活股票 0.9080%；存活偏差 +1.8180%
不可交易样本数 2 / 10
asset       date  limit_up  limit_down  suspended
    A 2024-01-03      True       False      False
    B 2024-01-03     False        True      False
质量断言： {'字段完整': True, '主键唯一': True, '价格为正': True, '无价格缺失': True, '资产数': 4, '日期数': 3}
版本快照 id： 9e330d2453ba
manifest： {"snapshot_id": "9e330d2453ba", "rows": 10, "checks": {"字段完整": true, "主键唯一": true, "价格为正": true, "无价格缺失": true, "资产数": 4, "日期数": 3}, "code_version": "data-eng-0.1"}



**读数**：全样本平均日收益 −0.91%，只用当前存活股票算出来 +0.908%，存活偏差 +1.818%——这就是“只看今天还在的股票”有多危险。面板里 10 条记录有 2 条不可交易（一条涨停、一条跌停），下游撮合应跳过它们。六项质量断言全部通过；把面板按 `(asset, date)` 排序后序列化，得到快照 id `9e330d2453ba`，manifest 记录了 10 行、六项检查与代码版本——这一版数据从此可追溯。

**常见错误**：

1. 用今天的指数成分回测十年前——存活偏差。
2. 停牌当 0 收益、涨跌停照常成交——高估收益与换手。
3. 质量断言失败仍然发布快照，下游拿到脏数据才发现。

---

## 本章小结

### 一张表

| 问题 | 症状 | 处理 |
|---|---|---|
| 供应商口径 | 字段含义不明 | 数据字典写清单位/主键/时间/复权 |
| 修订 | 三月与七月看到两个值 | `knowledge_date` + as-of |
| 除权 | 收益假跳空 | 后复权价格 + 成交量反向调整 |
| 换月 | 换月日假跳空 | 收益率拼接或比率调整 |
| 夜盘 | 归属交易日错 | 21:00 后算下一交易日 |
| 存活偏差 | 历史收益虚高 | 全样本 + 退市标记 |
| 停牌/涨跌停 | 高估成交率 | `tradable` 标记 |
| 版本 | 数据对不上 | hash 快照 + manifest |

### 八句话

1. 数据字典是入库前的第一道关。
2. 同一事实多个版本，用 `knowledge_date` 管。
3. as-of 查询同时过滤有效时间与知识时间。
4. 后复权适合回测，前复权适合看盘。
5. 换月拼收益率，不拼价格。
6. 夜盘 21:00 之后算下一交易日。
7. 存活偏差会让历史收益系统性虚高。
8. 断言 + 快照，让数据可追责。

---

## 本章习题

**1.** 最小数据字典至少要写清哪五项？

**2.** `report_period` 与 `knowledge_date` 有什么区别？as-of 查询该用哪个？

**3.** `merge_asof(..., direction="backward")` 的含义是什么？

**4.** 前复权与后复权的区别是什么？做长时间回测应该用哪个？

**5.** 10 送 10 后价格从 10 元变 5 元，后复权价与原始收益分别是多少？

**6.** 主力合约切换时直接拼接价格会发生什么？正确做法是什么？

**7.** 周五 21:30 的夜盘成交属于哪个交易日？为什么？

**8.** 存活偏差怎么产生？可以用什么方式检查？

<details><summary>参考答案</summary>

**1.** 字段名与单位、主键、时间语义（交易日/可见日）、复权口径、更新频率；再补缺失与异常规则。

**2.** `report_period` 是数据描述的期间（有效时间），`knowledge_date` 是数据开始可见的日期（知识时间）；as-of 查询用 `knowledge_date` 过滤，同时用 `report_period` 限制业务范围。

**3.** 对每个查询日期，向前找 `knowledge_date` 不晚于它的最近一条记录；找不到就返回缺失（不能取未来值）。

**4.** 后复权以最早价格为基准、历史价格不变；前复权以最新价格为基准、历史价格随新除权重算。长时间回测用后复权，避免复权因子随未来事件变化。

**5.** 后复权价保持除权前 10 元（后续再乘因子）；原始收益是 5/10−1=−50%，而真实收益应看复权后价格。

**6.** 换月日新旧合约价差会被算成收益，出现假跳空；正确做法是用收益率拼接，或用比率调整把新合约历史价格对齐。

**7.** 属于下周一。夜盘 21:00 之后属于下一交易日，周五夜盘对应下一个工作日。

**8.** 回测只取“当前仍存续/仍在指数里”的样本，退市与暴雷股票被剔除，历史收益虚高。检查方式：保留全样本并加退市标记，对比“全样本”与“存活样本”的收益差。

</details>

---

## 参考资料

- Kleppmann, M. *Designing Data-Intensive Applications*，第 3 章（存储与检索、版本化数据）。
- pandas `merge_asof` 文档：<https://pandas.pydata.org/docs/reference/api/pandas.merge_asof.html>
- 上海期货交易所、大连商品交易所交易时间与夜盘安排（品种细则以交易所官网为准）。
- 第 28 章（库存修订与当时可见的数据）；第 38 章（版本与 manifest）；第 40 章（双时间轴表结构）。
